In [ ]:
import pandas as pd
import os


In [ ]:
import statistics as s
from math import isnan
from itertools import filterfalse
import numpy as np

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 0)

In [ ]:
#getting csv with the viral concepts that will be chorts
concepts_csv = '/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/viral_cohort_patient_overlap_seasonal_nonseasonal_update1.csv'
viral_concept_df = pd.read_csv(concepts_csv)

In [ ]:
viral_concept_df.head(100)

In [ ]:
#nonseasonal DF

In [ ]:
nonseasonal_table = viral_concept_df.loc[:, ['concept_id','standard_concept_name','non_seasonal'] ]
ns = nonseasonal_table.loc[nonseasonal_table['non_seasonal'] == 'Y', :]
ns

In [ ]:
ns['concept_id'].count()

In [ ]:
def get_condition_summary(concept_id):
    """
    Fetches per-patient summary for the specified condition_concept_id,
    applying Cohort Builder UI filters (EHR + genomics, observation window,
    flat-events, standard concepts), and returns a DataFrame with one row per
    patient including:
      - condition_concept_id
      - standard_concept_name, standard_vocabulary
      - first and last diagnosis date for that concept
      - condition_type_concept_name, visit_occurrence_concept_name from first occurrence
      - visits_for_concept: count of unique visit_occurrence_id for the concept
      - visits_all_concepts: count of unique visits across all conditions
      - concept_count_ehr: count of distinct condition concepts in EHR
    """
    dataset = os.environ["WORKSPACE_CDR"]
    sql = f"""
    WITH
      ehr_genomics_patients AS (
        SELECT DISTINCT person_id
        FROM `{dataset}.cb_search_person`
        WHERE has_ehr_data = 1
          AND (
               has_whole_genome_variant      = 1
            OR has_lr_whole_genome_variant   = 1
            OR has_array_data                = 1
          )
      ),

      all_occ AS (
        SELECT
          co.person_id,
          co.condition_concept_id,
          co.visit_occurrence_id,
          co.condition_start_datetime,
          co.condition_end_datetime,
          co.condition_type_concept_id
        FROM `{dataset}.condition_occurrence` co
        JOIN ehr_genomics_patients eg
          ON co.person_id = eg.person_id

        -- only events that made it into the CB search table
        JOIN `{dataset}.cb_search_all_events` ev
          ON ev.person_id = co.person_id
         AND ev.concept_id = co.condition_concept_id
         AND DATE(co.condition_start_datetime) = ev.entry_date

        JOIN `{dataset}.concept` c_std
          ON co.condition_concept_id = c_std.concept_id
        WHERE c_std.standard_concept = 'S'
      ),

      spec_occ AS (
        SELECT *
        FROM all_occ
        WHERE condition_concept_id = {concept_id}
      ),

      detail AS (
        SELECT
          person_id,
          condition_concept_id,
          condition_start_datetime AS first_diag_date,
          condition_end_datetime   AS first_end_date,
          condition_type_concept_id,
          visit_occurrence_id,
          ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY condition_start_datetime) AS rn
        FROM spec_occ
      ),

      first_detail AS (
        SELECT
          d.person_id,
          d.condition_concept_id,
          d.first_diag_date,
          d.first_end_date,
          c_std.concept_name        AS standard_concept_name,
          c_std.vocabulary_id       AS standard_vocabulary,
          c_type.concept_name       AS condition_type_concept_name,
          vis_evt.concept_name      AS visit_occurrence_concept_name
        FROM detail d
        JOIN `{dataset}.concept` c_std
          ON d.condition_concept_id = c_std.concept_id
        LEFT JOIN `{dataset}.concept` c_type
          ON d.condition_type_concept_id = c_type.concept_id
        LEFT JOIN `{dataset}.visit_occurrence` v
          ON d.visit_occurrence_id = v.visit_occurrence_id
        LEFT JOIN `{dataset}.concept` vis_evt
          ON v.visit_concept_id = vis_evt.concept_id
        WHERE d.rn = 1
      ),

      spec_metrics AS (
        SELECT
          person_id,
          MIN(condition_start_datetime) AS first_diag_date,
          MAX(condition_start_datetime) AS last_diag_date,
          COUNT(DISTINCT visit_occurrence_id) AS visits_for_concept
        FROM spec_occ
        GROUP BY person_id
      ),

      allv AS (
        SELECT
          person_id,
          COUNT(DISTINCT visit_occurrence_id) AS visits_all_concepts
        FROM all_occ
        GROUP BY person_id
      ),

      conc AS (
        SELECT
          person_id,
          COUNT(DISTINCT condition_concept_id) AS concept_count_ehr
        FROM all_occ
        GROUP BY person_id
      )

    SELECT
      fd.person_id,
      fd.condition_concept_id,
      fd.standard_concept_name,
      fd.standard_vocabulary,
      fd.first_diag_date,
      sm.last_diag_date,
      fd.condition_type_concept_name,
      fd.visit_occurrence_concept_name,
      sm.visits_for_concept,
      av.visits_all_concepts,
      cc.concept_count_ehr
    FROM first_detail fd
    JOIN spec_metrics sm  ON fd.person_id = sm.person_id
    LEFT JOIN allv av       ON fd.person_id = av.person_id
    LEFT JOIN conc cc       ON fd.person_id = cc.person_id
    """

    df = pd.read_gbq(
        sql,
        project_id=os.environ.get("BIGQUERY_PROJECT"),
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook",
    )
    return df

In [ ]:
def demographics_table():
    """
    Fetch person demographics rows for a single concept_id,
    using your original SQL structure and injecting concept_id directly.
    """
    dataset = os.environ["WORKSPACE_CDR"]

    demographics_sql  = f"""
    SELECT
        person.person_id,
        
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
   
        p_race_concept.concept_name as race,
    
        p_ethnicity_concept.concept_name as ethnicity,
    
        p_sex_at_birth_concept.concept_name as sex_at_birth,
      
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `{dataset}.person` person 
    LEFT JOIN
        `{dataset}.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `{dataset}.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_array_data = 1 ) )"""


    demographics_df = pd.read_gbq(
        demographics_sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook"
    )

    return demographics_df

In [ ]:
def merge_data_table(): 
    
    cohort_dict = {}
    
    nonseasonal_table = viral_concept_df.loc[:, ['concept_id','standard_concept_name','non_seasonal']]
    ns = nonseasonal_table.loc[nonseasonal_table['non_seasonal'] == 'Y', :]
    
    nonseasonal_concept_id = ns['concept_id'].to_list()
  
    
    nonseasonal_concept_name = ns['standard_concept_name'].to_list()
 
    
    # call demo and socio function for table and drop duplicates by person_id: one-row-per-patient tables once
    demo = demographics_table().drop_duplicates('person_id')
    
    for concept_id, concept_name in zip(nonseasonal_concept_id, nonseasonal_concept_name):
    
        cond = get_condition_summary(concept_id)
        
        merged = cond.merge(demo,  on='person_id', how='left')
             
        
        
        cohort_dict[(concept_id, concept_name)] = merged
       

    return cohort_dict

In [ ]:
ns_df = merge_data_table()

In [ ]:
ns_df

In [ ]:
def merge_race_ethnicity_data(new_column):
    
    if new_column["ethnicity"] == "Hispanic or Latino":
        return new_column["ethnicity"]
    else:
        return new_column["race"]
    
for key, table in ns_df.items():
    
    table["updated_race"] = table.apply(merge_race_ethnicity_data, axis=1)
    
     

In [ ]:
import pandas as pd
import pickle
from pathlib import Path

# 1) Suppose `ns_df` is your dict of DataFrames
#    Example: ns_df = {'a': df1, 'b': df2, ...}

# 2) Choose a workspace folder for persistence
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_cohort_data_table.pkl')

# 3) Save the entire dict in one go
with open(out_file, 'wb') as f:
    pickle.dump(ns_df, f)

print(f"Saved {len(ns_df)} DataFrames to {out_file}")

In [ ]:
import pandas as pd
import pickle
from pathlib import Path

# …later, in any notebook in the same workspace…
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_cohort_data_table.pkl')

# 4) Reload with one line:
with open(out_file, 'rb') as f:
    ns_df = pickle.load(f)

#print("Reloaded keys:", list(ns_df.keys()))

In [ ]:
ns_df

In [ ]:
def get_ns_binning(nonseasonal_df):
    
    ns_cohort_bin_dict = {}
    
    for key, table in nonseasonal_df.items():
        
        
        race_count = table["updated_race"].value_counts()
        df1 = pd.DataFrame(race_count)
        df1['condition_concept_id'] = key[0] #make a concept_id column with key values
        df1['condition_concept_name'] = key[1]
        df2 = df1.reset_index() #make race a column
        
        ns_cohort_bin_dict[key] = df2
        
        
        
    ns_table = pd.concat(ns_cohort_bin_dict.values(), axis = 0, ignore_index = True)
    new_order = ['condition_concept_id', 'standard_concept_name','updated_race', 'count']
    ns_table = ns_table[new_order]
    
        
        
    return df1
    
    

In [ ]:
def get_ns_binning(nonseasonal_df):
    
    ns_cohort_bin_dict = {}
    
    for key, table in nonseasonal_df.items():
        
        
        race_count = table["updated_race"].value_counts()
        df1 = pd.DataFrame(race_count)
        df1['condition_concept_id'] = key[0] #make a concept_id column with key values
        df1['standard_concept_name'] = key[1]
        df2 = df1.reset_index() #make race a column
        
        ns_cohort_bin_dict[key] = df2
        
        
        
    ns_table = pd.concat(ns_cohort_bin_dict.values(), axis = 0, ignore_index = True)
    new_order = ['concept_id', 'concept_name','updated_race', 'count']
    ns_table = ns_table[new_order]
    final = ns_table.rename(columns={'concept_id': 'condition_concept_id', 'concept_name': 'standard_concept_name'})
    
        
        
    return final
    
    

In [ ]:
ns_race_bin_counts = get_ns_binning(ns_df)
ns_race_bin_counts

In [ ]:
ns_race_bin_counts['updated_race'].unique()

In [ ]:
# 1) List of races to exclude
to_drop = [
    'PMI: Skip',
    'None of these',
    'American Indian or Alaska Native',
    'I prefer not to answer'
]

# 2) Keep only rows whose updated_race is not in that list
df_filtered1 = ns_race_bin_counts[~ns_race_bin_counts['updated_race'].isin(to_drop)]

In [ ]:
df_filtered1

In [ ]:
df_filtered = df_filtered1.copy()

In [ ]:
import pandas as pd
import numpy as np

# assume your DataFrame is called df and looks like:
#   concept_id  concept_name    updated_race  count
# 0      440029  Viral disease   White         18913
# …      …       …               …             …

# 1) Flag which race‐rows meet the ≥100 threshold
df_filtered['race_ge_100'] = df_filtered['count'] >= 100

# 2) Count, per concept, how many races are ≥100
concept_counts = (
    df_filtered 
      .groupby(['condition_concept_id','standard_concept_name'], as_index=False)
      .agg(n_eligible_races=('race_ge_100','sum'))
)

# 3) Bin each concept:
#    B1: ≥2 races ≥100
#    B2: exactly 1 race ≥100
#    A1:  0 races ≥100
conds = [
    concept_counts['n_eligible_races'] >= 2,
    concept_counts['n_eligible_races'] == 1,
    concept_counts['n_eligible_races'] == 0,
]
choices = ['B1','B2','A1']
concept_counts['bin'] = np.select(conds, choices, default='A1')

# 4) (Optional) Merge the bin back onto your original rows
df_binned = df_filtered.merge(
    concept_counts[['standard_concept_name','bin']],
    on='standard_concept_name',
    how='left'
)

# 5) Inspect
print(concept_counts['bin'].value_counts())


In [ ]:
for grp in ['B1','B2','A1']:
    members = concept_counts.loc[concept_counts['bin']==grp, ['condition_concept_id','standard_concept_name']]
    print(f"\n=== {grp} ({len(members)} concepts) ===")
    print(members)

In [ ]:
concept_counts

In [ ]:
df_binned.head(50)

In [ ]:
#getting person_Id from filtering

In [ ]:
master = pd.concat(
    [
        df.assign(
            concept_id=cid_name[0],
            concept_name=cid_name[1]
        )
        for cid_name, df in ns_df.items()
    ],
    ignore_index=True
)



# 1) Filter your binned summary to only keep races with ≥100 people
df_binned_filtered = df_binned[df_binned['race_ge_100']]

# 2) Merge in person_id (only for those high-N rows)
joined = df_binned_filtered.merge(
    master[
      ['condition_concept_id',
       'standard_concept_name',
       'updated_race',
       'person_id']
    ],
    on=['condition_concept_id',
        'standard_concept_name',
        'updated_race'],
    how='left'
)

# 3) Aggregate each (concept, race, bin) into a list of person_ids
person_lists = (
    joined
    .groupby(
        ['condition_concept_id',
         'standard_concept_name',
         'updated_race',
         'bin'],
        as_index=False
    )['person_id']
    .apply(list)
    .rename(columns={'person_id':'person_ids'})
)

# 4) Split into a dict of DataFrames by bin
dfs_by_bin = {
    b: df_bin.reset_index(drop=True)
    for b, df_bin in person_lists.groupby('bin')
}

# ─── (Optional) further nest by concept within each bin ───────────────────────
nested = {
    b: {
        cid: subdf.drop(columns='bin').reset_index(drop=True)
        for cid, subdf in dfs_by_bin[b].groupby('condition_concept_id')
    }
    for b in dfs_by_bin
}

# ─── Usage ─────────────────────────────────────────────────────────────────────
# DataFrame for bin "A1":
df_B1 = dfs_by_bin['B1']
df_B2 = dfs_by_bin['B2']





# Now dfs_by_bin['A1'], dfs_by_bin['B1'], dfs_by_bin['B2'], …
# each only contain rows where race_ge_100 was True.


In [ ]:
df_B1['standard_concept_name'].unique()

In [ ]:
df_B2

In [ ]:
# ─── Inputs ────────────────────────────────────────────────────────────────────
# df_binned_filtered: your summary DF already filtered to race_ge_100 == True,
#   with columns ['condition_concept_id','standard_concept_name','updated_race',…]
# ns_df: dict mapping (condition_concept_id, standard_concept_name) → the full person-level DF
#
# ─── Build your triple-keyed dict ──────────────────────────────────────────────
filtered_by_triple = {}
for row in df_binned_filtered.itertuples(index=False):
    cid, name, race = (
        row.condition_concept_id,
        row.standard_concept_name,
        row.updated_race
    )
    # grab the master DF for that concept
    df_persons = ns_df[(cid, name)]
    # filter it down to just that race
    df_race = df_persons[df_persons['updated_race'] == race].copy()
    # store under the triple key
    filtered_by_triple[(cid, name, race)] = df_race

# ─── Now e.g. ──────────────────────────────────────────────────────────────────
# get the DataFrame for (440029, "Viral disease", "Asian"):
df_viral_asian = filtered_by_triple[(440029, "Viral disease", "Asian")]


In [ ]:
df_viral_asian = filtered_by_triple[(440029, "Viral disease", "Asian")]
df_viral_asian

In [ ]:
filtered_by_triple.values()

In [ ]:
import pickle
from pathlib import Path

# 1) Suppose `ns_vax_df` is your dict of DataFrames
#    Example: ns_vax_df = {'a': df1, 'b': df2, ...}

# 2) Choose a workspace folder for persistence
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_B1_B2_data_table.pkl')

# 3) Save the entire dict in one go
with open(out_file, 'wb') as f:
    pickle.dump(filtered_by_triple, f)

print(f"Saved {len(filtered_by_triple)} DataFrames to {out_file}")

In [ ]:
import pandas as pd
import pickle
from pathlib import Path

# …later, in any notebook in the same workspace…
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_B1_B2_data_table.pkl')

# 4) Reload with one line:
with open(out_file, 'rb') as f:
    race_only_df = pickle.load(f)

#print("Reloaded keys:", list(ns_df.keys()))

In [ ]:
race_only_df.keys()

In [ ]:
for key in race_only_df.keys():
    print(key)